# Start

> HEALPix-based spatial aggregation for planetary science data

## Features

- **Batch Processing**: Classical split-apply-combine for static datasets
- **Streaming Aggregation**: Incremental statistics for growing datasets
- **Memory Efficient**: Process datasets larger than RAM
- **Native Resolution Mosaics**: Fine-grained HEALPix grids
- **Flexible Statistics**: Mean, median, std, MAD, robust_std, percentiles

## Installation

```bash
pip install healpyxel
```

### Optional Dependencies

```bash
# For percentile tracking
pip install healpyxel[tdigest]

# For parallel processing
pip install healpyxel[dask]

# For efficient I/O
pip install healpyxel[duckdb]

# All extras
pip install healpyxel[dev,tdigest,dask,duckdb]
```

## Quick Start

### Batch Processing

```bash
# 1. Generate HEALPix sidecar (SPLIT)
healpyxel-sidecar --input observations.parquet --nside 64 128 --mode fuzzy

# 2. Aggregate by HEALPix cells (APPLY)
healpyxel-aggregate observations.parquet --sidecar-index 0 \
  --columns r750 r950 --aggs median robust_std --output results.parquet
```

### Streaming Processing

```bash
# Day 1: Initialize accumulator
healpyxel-accumulate --input day001.parquet \
  --columns r750 r950 --state-output state_v001.parquet

# Day 2+: Incremental updates
healpyxel-accumulate --input day002.parquet \
  --columns r750 r950 \
  --state-input state_v001.parquet --state-output state_v002.parquet

# Finalize to statistics
healpyxel-finalize --state state_v030.parquet --output mosaic.parquet \
  --percentiles 25 50 75 --densify --nside 512
```

## Python API

```python
from healpyxel import sidecar, aggregate, accumulator, finalize

# Generate sidecar
sidecar_df = sidecar.generate(
    gdf,
    nside=64,
    mode='fuzzy'
)

# Aggregate
result = aggregate.by_sidecar(
    original=df,
    sidecar=sidecar_df,
    value_columns=['r750', 'r950'],
    aggs=['median', 'robust_std']
)
```

## Documentation

See the [full documentation](https://mariodamore.github.io/healpyxel) for:

- Detailed tutorials
- API reference
- Performance optimization tips
- Example workflows

## Developed for MESSENGER/MASCS

This package was developed to process spectral observations from the MESSENGER/MASCS instrument studying Mercury's surface. The workflow handles:

- Millions of observations with complex footprint geometries
- Multi-spectral reflectance data (VIS + NIR)
- Streaming data from ongoing missions
- Native resolution mosaics (sub-footprint sampling)

While designed for MASCS, healpyxel is general-purpose and works with any planetary science dataset in GeoParquet format.

### Useful Healpix data for those missions



In [ ]:
import healpy as hp
import pandas as pd
import numpy as np

# Mean planetary radii (km)
R_MERCURY = 2439.7
R_MOON = 1737.4
R_VENUS = 6051.8

#| export
nsides = 2 ** np.arange(1, 13, dtype=np.int64)
npix = hp.nside2npix(nsides)
cell_ang_rad = np.sqrt(4 * np.pi / npix)
cell_ang_deg = np.degrees(cell_ang_rad)

df_resolution = pd.DataFrame({
    "nside": nsides,
    "Number of Cells": npix,
    "Cell Angular Size": cell_ang_deg,
    "Mercury Cell Size": cell_ang_rad * R_MERCURY,
    "Moon Cell Size": cell_ang_rad * R_MOON,
    "Venus Cell Size": cell_ang_rad * R_VENUS,
})

#| export
def format_resolution_table(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the resolution table with int columns formatted with thousand separators and floats rounded to 3 decimals."""
    df_fmt = df.copy()
    int_cols = df_fmt.select_dtypes(include="int64").columns
    float_cols = df_fmt.select_dtypes(include="float64").columns
    for col in int_cols:
        df_fmt[col] = df_fmt[col].map(lambda x: f"{x:,}")
    for col in float_cols:
        df_fmt[col] = df_fmt[col].round(3)
    return df_fmt.set_index("nside")

# Test: Ensure formatting is correct
fmt = format_resolution_table(df_resolution)

print(format_resolution_table(df_resolution).to_markdown()
)

| nside   | Number of Cells   |   Cell Angular Size |   Mercury Cell Size |   Moon Cell Size |   Venus Cell Size |
|:--------|:------------------|--------------------:|--------------------:|-----------------:|------------------:|
| 2       | 48                |              29.316 |            1248.31  |          888.964 |          3096.48  |
| 4       | 192               |              14.658 |             624.153 |          444.482 |          1548.24  |
| 8       | 768               |               7.329 |             312.076 |          222.241 |           774.121 |
| 16      | 3,072             |               3.665 |             156.038 |          111.12  |           387.061 |
| 32      | 12,288            |               1.832 |              78.019 |           55.56  |           193.53  |
| 64      | 49,152            |               0.916 |              39.01  |           27.78  |            96.765 |
| 128     | 196,608           |               0.458 |              19.505 |     

## License

Apache 2.0